# Notebook 03 — Match-Event Context & Footballing-Intensity Proxies (StatsBomb Open Data)

## Goal
Build a **per-player, per-match event feature table** from StatsBomb open-data that captures footballing-intensity proxies relevant to injury risk (duels, pressures, fouls suffered, carries, shots, defensive actions, substitutions, position changes, role usage).

The output is:
1. A clean `player_match_event_features` DataFrame (one row per player per match).
2. A short **Coverage & Join Memo** assessing external-join difficulty with Transfermarkt.

---

## Section 1 — Setup & Imports

In [1]:
import json
import os
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid", font_scale=0.9)

# ── Paths ────────────────────────────────────────────────────────────────
DATA_ROOT = Path("../data/open-data/data")
EVENTS_DIR = DATA_ROOT / "events"
LINEUPS_DIR = DATA_ROOT / "lineups"
MATCHES_DIR = DATA_ROOT / "matches"
THREESIXTY_DIR = DATA_ROOT / "three-sixty"
COMPETITIONS_FILE = DATA_ROOT / "competitions.json"

print("Event files     :", len(list(EVENTS_DIR.glob("*.json"))))
print("Lineup files    :", len(list(LINEUPS_DIR.glob("*.json"))))
print("Three-sixty files:", len(list(THREESIXTY_DIR.glob("*.json"))))
print("Match dirs      :", sorted([d.name for d in MATCHES_DIR.iterdir() if d.is_dir()]))

Event files     : 3464
Lineup files    : 3464
Three-sixty files: 326
Match dirs      : ['11', '116', '12', '1238', '1267', '1470', '16', '2', '223', '35', '37', '43', '44', '49', '53', '55', '7', '72', '81', '87', '9']


## Section 2 — Load Competitions & Matches Metadata

Build an index of every match available in the dataset so we can later join match-level context (competition, date, teams) onto the event features.

In [2]:
# ── Load competitions catalogue ──────────────────────────────────────────
with open(COMPETITIONS_FILE) as f:
    competitions = json.load(f)
comp_df = pd.DataFrame(competitions)
print(f"Competitions catalogue: {len(comp_df)} competition-season rows")
print(comp_df[["competition_id", "competition_name", "season_id", "season_name",
               "country_name", "competition_gender"]].head(10))
print()

# ── Load all matches into one DataFrame ──────────────────────────────────
match_rows = []
for comp_dir in sorted(MATCHES_DIR.iterdir()):
    if not comp_dir.is_dir():
        continue
    for season_file in sorted(comp_dir.glob("*.json")):
        with open(season_file) as f:
            matches = json.load(f)
        for m in matches:
            match_rows.append({
                "match_id": m["match_id"],
                "match_date": m["match_date"],
                "kick_off": m.get("kick_off"),
                "competition_id": m["competition"]["competition_id"],
                "competition_name": m["competition"]["competition_name"],
                "season_id": m["season"]["season_id"],
                "season_name": m["season"]["season_name"],
                "home_team_id": m["home_team"]["home_team_id"],
                "home_team_name": m["home_team"]["home_team_name"],
                "away_team_id": m["away_team"]["away_team_id"],
                "away_team_name": m["away_team"]["away_team_name"],
                "home_score": m["home_score"],
                "away_score": m["away_score"],
                "match_week": m.get("match_week"),
            })

matches_df = pd.DataFrame(match_rows)
matches_df["match_date"] = pd.to_datetime(matches_df["match_date"])
print(f"Total matches indexed: {len(matches_df)}")
print(f"Date range: {matches_df['match_date'].min().date()} → {matches_df['match_date'].max().date()}")
print(f"Competitions: {matches_df['competition_name'].nunique()}")
print()
matches_df.groupby(["competition_name", "season_name"]).size().reset_index(name="n_matches").head(15)

Competitions catalogue: 75 competition-season rows
   competition_id        competition_name  season_id season_name country_name  \
0               9           1. Bundesliga        281   2023/2024      Germany   
1               9           1. Bundesliga         27   2015/2016      Germany   
2            1267  African Cup of Nations        107        2023       Africa   
3              16        Champions League          4   2018/2019       Europe   
4              16        Champions League          1   2017/2018       Europe   
5              16        Champions League          2   2016/2017       Europe   
6              16        Champions League         27   2015/2016       Europe   
7              16        Champions League         26   2014/2015       Europe   
8              16        Champions League         25   2013/2014       Europe   
9              16        Champions League         24   2012/2013       Europe   

  competition_gender  
0               male  
1          

,competition_name,season_name,n_matches
0,1. Bundesliga,2015/2016,306
1,1. Bundesliga,2023/2024,34
2,African Cup of Nations,2023,52
3,Champions League,1970/1971,1
4,Champions League,1971/1972,1
5,Champions League,1972/1973,1
6,Champions League,1999/2000,1
7,Champions League,2003/2004,1
8,Champions League,2004/2005,1
9,Champions League,2006/2007,1


## Section 3 — Parse Lineups (Player Roster per Match)

Each lineup file gives us the players who were in the squad for a match, their positions, and cards received. This creates the **player × match** grain we need.

In [3]:
lineup_rows = []
for lineup_file in sorted(LINEUPS_DIR.glob("*.json")):
    match_id = int(lineup_file.stem)
    with open(lineup_file) as f:
        teams = json.load(f)
    for team in teams:
        team_id = team["team_id"]
        team_name = team["team_name"]
        for player in team["lineup"]:
            # positions list can record multiple positions if player shifted
            positions = player.get("positions", [])
            n_positions = len(positions)
            primary_position = positions[0]["position"] if positions else None
            lineup_rows.append({
                "match_id": match_id,
                "team_id": team_id,
                "team_name": team_name,
                "player_id": player["player_id"],
                "player_name": player["player_name"],
                "jersey_number": player.get("jersey_number"),
                "primary_position": primary_position,
                "n_positions_played": n_positions,
            })

lineup_df = pd.DataFrame(lineup_rows)
print(f"Lineup rows (player × match): {len(lineup_df):,}")
print(f"Unique players: {lineup_df['player_id'].nunique():,}")
print(f"Unique matches: {lineup_df['match_id'].nunique():,}")
print()
lineup_df.head()

Lineup rows (player × match): 131,901
Unique players: 10,803
Unique matches: 3,464



,match_id,team_id,team_name,player_id,player_name,jersey_number,primary_position,n_positions_played
0,15946,217,Barcelona,3109,Malcom Filipe Silva de Oliveira,14,NaN,0
1,15946,217,Barcelona,3501,Philippe Coutinho Correia,7,Left Center Midfield,3
2,15946,217,Barcelona,5203,Sergio Busquets i Burgos,5,Right Defensive Midfield,2
3,15946,217,Barcelona,5211,Jordi Alba Ramos,18,Left Back,1
4,15946,217,Barcelona,5213,Gerard Piqué Bernabéu,3,Right Center Back,1


## Section 4 — Parse All Events & Build Raw Event Table

We iterate over every event file and extract the fields relevant to intensity proxies. The event types we care about:

| Event type | Injury-risk relevance |
|---|---|
| **Duel** | Physical contact intensity (aerial + tackle) |
| **Pressure** | High-intensity defensive effort |
| **Foul Won** | Getting fouled — direct trauma risk |
| **Foul Committed** | Committing fouls — aggression proxy |
| **Carry** | Ball-carrying distance / volume |
| **Shot** | Explosive actions |
| **Clearance** | Defensive effort |
| **Interception** | Defensive effort |
| **Ball Recovery** | Defensive effort |
| **Block** | Defensive effort |
| **Dribble** | Acceleration / change-of-direction |
| **Substitution** | Game-time / early removal |
| **Tactical Shift** | Formation change |
| **50/50** | Contested ball |

In [ ]:
# ── Event types of interest for intensity features ───────────────────────
INTENSITY_TYPES = {
    "Duel", "Pressure", "Foul Won", "Foul Committed",
    "Carry", "Shot", "Clearance", "Interception",
    "Ball Recovery", "Block", "Dribble", "Dribbled Past",
    "Substitution", "Tactical Shift", "50/50",
    "Bad Behaviour", "Miscontrol", "Dispossessed",
}

def parse_event_file(filepath):
    """Parse a single match event JSON into a list of flat dicts."""
    match_id = int(filepath.stem)
    with open(filepath) as f:
        events = json.load(f)

    rows = []
    for e in events:
        etype = e["type"]["name"]
        player = e.get("player")
        if player is None:
            continue  # skip team-level events without a player

        row = {
            "match_id": match_id,
            "event_id": e["id"],
            "period": e["period"],
            "minute": e["minute"],
            "second": e["second"],
            "event_type": etype,
            "player_id": player["id"],
            "player_name": player["name"],
            "team_id": e["team"]["id"],
            "team_name": e["team"]["name"],
            "position": e.get("position", {}).get("name"),
            "duration": e.get("duration"),
            "under_pressure": e.get("under_pressure", False),
        }

        # ── Duel sub-type (aerial vs tackle) ────────────────────────
        if etype == "Duel":
            duel_info = e.get("duel", {})
            row["duel_type"] = duel_info.get("type", {}).get("name")
            row["duel_outcome"] = duel_info.get("outcome", {}).get("name")

        # ── Foul: card info ──────────────────────────────────────────
        if etype == "Foul Committed":
            fc = e.get("foul_committed", {})
            card = fc.get("card", {})
            row["card"] = card.get("name")
            row["foul_offensive"] = fc.get("offensive", False)
        if etype == "Foul Won":
            fw = e.get("foul_won", {})
            row["foul_won_defensive"] = fw.get("defensive", False)

        # ── Carry distance ───────────────────────────────────────────
        if etype == "Carry":
            loc = e.get("location")
            end = e.get("carry", {}).get("end_location")
            if loc and end:
                row["carry_distance"] = np.sqrt(
                    (end[0] - loc[0])**2 + (end[1] - loc[1])**2
                )

        # ── Shot xG ──────────────────────────────────────────────────
        if etype == "Shot":
            shot_info = e.get("shot", {})
            row["shot_statsbomb_xg"] = shot_info.get("statsbomb_xg")
            row["shot_outcome"] = shot_info.get("outcome", {}).get("name")

        # ── Substitution details ─────────────────────────────────────
        if etype == "Substitution":
            sub_info = e.get("substitution", {})
            row["sub_replacement_id"] = sub_info.get("replacement", {}).get("id")
            row["sub_replacement_name"] = sub_info.get("replacement", {}).get("name")
            row["sub_outcome"] = sub_info.get("outcome", {}).get("name")

        # ── Pressure counterpress flag ───────────────────────────────
        if etype == "Pressure":
            row["counterpress"] = e.get("counterpress", False)

        rows.append(row)
    return rows


# ── Parse all event files ────────────────────────────────────────────────
all_event_rows = []
event_files = sorted(EVENTS_DIR.glob("*.json"))
n_files = len(event_files)

for i, ef in enumerate(event_files):
    all_event_rows.extend(parse_event_file(ef))
    if (i + 1) % 500 == 0:
        print(f"  parsed {i+1}/{n_files} files …")

events_df = pd.DataFrame(all_event_rows)
print(f"\nTotal player-event rows: {len(events_df):,}")
print(f"Unique matches: {events_df['match_id'].nunique():,}")
print(f"Unique players: {events_df['player_id'].nunique():,}")
print(f"Event types: {events_df['event_type'].nunique()}")
print()
events_df["event_type"].value_counts()

  parsed 500/3464 files …
  parsed 1000/3464 files …
  parsed 1500/3464 files …
  parsed 2000/3464 files …
  parsed 2500/3464 files …
  parsed 3000/3464 files …


## Section 5 — Aggregate: Per-Player, Per-Match Event Features

Pivot the event-level table into a one-row-per-player-per-match feature table. Each column represents a count, sum, or derived metric for that player in that match.

In [ ]:
def build_player_match_features(events_df):
    """
    Aggregate raw event rows into per-player, per-match intensity features.
    Returns a DataFrame with one row per (player_id, match_id).
    """
    grp = events_df.groupby(["match_id", "player_id", "player_name", "team_id", "team_name"])

    # ── Count-based features ─────────────────────────────────────────────
    def _count(df, etype):
        return (df["event_type"] == etype).sum()

    agg = grp.apply(lambda df: pd.Series({
        # Physical intensity
        "n_duels":            _count(df, "Duel"),
        "n_aerial_duels":     ((df["event_type"] == "Duel") & df["duel_type"].str.contains("Aerial", na=False)).sum(),
        "n_tackle_duels":     ((df["event_type"] == "Duel") & df["duel_type"].str.contains("Tackle", na=False)).sum(),
        "n_duels_won":        ((df["event_type"] == "Duel") & df["duel_outcome"].isin(["Won", "Success In Play", "Success Out"])).sum() if "duel_outcome" in df.columns else 0,
        "n_pressures":        _count(df, "Pressure"),
        "n_counterpresses":   ((df["event_type"] == "Pressure") & (df["counterpress"] == True)).sum(),
        "pressure_duration_s": df.loc[df["event_type"] == "Pressure", "duration"].sum(),

        # Fouls / aggression
        "n_fouls_won":        _count(df, "Foul Won"),
        "n_fouls_committed":  _count(df, "Foul Committed"),
        "n_cards":            df.loc[df["event_type"] == "Foul Committed", "card"].notna().sum(),
        "n_yellow_cards":     (df["card"] == "Yellow Card").sum() if "card" in df.columns else 0,

        # Ball carrying / progression
        "n_carries":          _count(df, "Carry"),
        "total_carry_dist":   df.loc[df["event_type"] == "Carry", "carry_distance"].sum(),
        "mean_carry_dist":    df.loc[df["event_type"] == "Carry", "carry_distance"].mean(),

        # Attacking output
        "n_shots":            _count(df, "Shot"),
        "total_xg":           df.loc[df["event_type"] == "Shot", "shot_statsbomb_xg"].sum(),
        "n_dribbles":         _count(df, "Dribble"),
        "n_dribbled_past":    _count(df, "Dribbled Past"),

        # Defensive actions
        "n_clearances":       _count(df, "Clearance"),
        "n_interceptions":    _count(df, "Interception"),
        "n_ball_recoveries":  _count(df, "Ball Recovery"),
        "n_blocks":           _count(df, "Block"),

        # Errors / turnovers
        "n_miscontrols":      _count(df, "Miscontrol"),
        "n_dispossessed":     _count(df, "Dispossessed"),

        # Composite / derived
        "n_defensive_actions": (df["event_type"].isin(["Clearance", "Interception", "Ball Recovery", "Block"])).sum(),
        "n_events_under_pressure": df["under_pressure"].sum(),
        "total_event_count":  len(df),
    })).reset_index()

    return agg


player_match_df = build_player_match_features(events_df)
print(f"Player-match feature table: {player_match_df.shape[0]:,} rows × {player_match_df.shape[1]} columns")
print(f"Unique players: {player_match_df['player_id'].nunique():,}")
print(f"Unique matches: {player_match_df['match_id'].nunique():,}")
player_match_df.head()

## Section 6 — Add Substitution, Position-Change & Role Features

Substitution info (was the player subbed off? in which minute?) and position changes during a match are additional intensity/fatigue proxies.

In [ ]:
# ── Substitution features ─────────────────────────────────────────────────
# A "Substitution" event is logged for the player being REPLACED (subbed off).
subs = events_df[events_df["event_type"] == "Substitution"][
    ["match_id", "player_id", "minute", "sub_outcome"]
].rename(columns={"minute": "sub_off_minute", "sub_outcome": "sub_off_reason"})
subs["was_subbed_off"] = True

player_match_df = player_match_df.merge(subs, on=["match_id", "player_id"], how="left")
player_match_df["was_subbed_off"] = player_match_df["was_subbed_off"].fillna(False)

print(f"Players subbed off: {player_match_df['was_subbed_off'].sum():,} "
      f"({100 * player_match_df['was_subbed_off'].mean():.1f}%)")
if "sub_off_reason" in player_match_df.columns:
    print(player_match_df["sub_off_reason"].value_counts(dropna=False).head())
print()

# ── Position diversity (from events, not lineups) ────────────────────────
# How many distinct positions did the player occupy across their events in this match?
pos_counts = (
    events_df.dropna(subset=["position"])
    .groupby(["match_id", "player_id"])["position"]
    .nunique()
    .reset_index()
    .rename(columns={"position": "n_distinct_positions"})
)

player_match_df = player_match_df.merge(pos_counts, on=["match_id", "player_id"], how="left")
player_match_df["n_distinct_positions"] = player_match_df["n_distinct_positions"].fillna(0).astype(int)

print(f"Position diversity: median={player_match_df['n_distinct_positions'].median():.0f}, "
      f"max={player_match_df['n_distinct_positions'].max()}")

# ── Role from lineup (primary position) ──────────────────────────────────
player_match_df = player_match_df.merge(
    lineup_df[["match_id", "player_id", "primary_position", "n_positions_played"]],
    on=["match_id", "player_id"],
    how="left",
    suffixes=("", "_lineup"),
)

print(f"Primary position coverage: {player_match_df['primary_position'].notna().mean():.1%}")
print()
player_match_df[["player_name", "match_id", "was_subbed_off", "sub_off_minute",
                  "n_distinct_positions", "primary_position"]].head(10)

## Section 7 — Merge Match Metadata

Attach competition, date, season, and team info to the player-match features so we can later filter, stratify, and join with external data.

In [ ]:
# Merge match-level context
player_match_df = player_match_df.merge(
    matches_df[["match_id", "match_date", "competition_id", "competition_name",
                "season_id", "season_name", "home_team_id", "away_team_id",
                "home_score", "away_score", "match_week"]],
    on="match_id",
    how="left",
)

print(f"Shape after match merge: {player_match_df.shape}")
print(f"Match date coverage: {player_match_df['match_date'].notna().mean():.1%}")
print(f"Competitions covered: {player_match_df['competition_name'].nunique()}")
print()

# Summary by competition
comp_summary = (
    player_match_df.groupby("competition_name")
    .agg(
        n_matches=("match_id", "nunique"),
        n_players=("player_id", "nunique"),
        n_rows=("player_id", "size"),
        date_min=("match_date", "min"),
        date_max=("match_date", "max"),
    )
    .sort_values("n_matches", ascending=False)
)
comp_summary

## Section 8 — Exploratory Distributions of Key Intensity Features

Quick look at the distributions to sanity-check the features and spot any outliers or data issues.

In [ ]:
feature_cols = [
    "n_duels", "n_pressures", "n_fouls_won", "n_fouls_committed",
    "n_carries", "total_carry_dist", "n_shots", "n_dribbles",
    "n_defensive_actions", "n_events_under_pressure",
    "total_event_count", "n_counterpresses",
]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, col in zip(axes.flat, feature_cols):
    data = player_match_df[col].dropna()
    ax.hist(data, bins=40, edgecolor="white", alpha=0.8)
    ax.set_title(col, fontsize=10)
    ax.axvline(data.median(), color="red", ls="--", lw=1, label=f"med={data.median():.1f}")
    ax.legend(fontsize=7)
fig.suptitle("Per-Player, Per-Match Feature Distributions", fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
# ── Summary statistics table ──────────────────────────────────────────────
numeric_cols = player_match_df.select_dtypes(include=[np.number]).columns.tolist()
# Exclude ID columns
numeric_cols = [c for c in numeric_cols if c not in
                {"match_id", "player_id", "team_id", "competition_id",
                 "season_id", "home_team_id", "away_team_id",
                 "home_score", "away_score", "match_week",
                 "sub_replacement_id", "jersey_number"}]

stats = player_match_df[numeric_cols].describe().T
stats["non_zero_%"] = ((player_match_df[numeric_cols] != 0).mean() * 100).round(1)
stats = stats[["count", "mean", "std", "min", "50%", "max", "non_zero_%"]]
stats.round(2)

## Section 9 — Correlation Between Intensity Features

Check which features are highly correlated (redundant) vs. which capture independent information.

In [ ]:
corr_cols = [
    "n_duels", "n_aerial_duels", "n_tackle_duels",
    "n_pressures", "n_counterpresses", "pressure_duration_s",
    "n_fouls_won", "n_fouls_committed",
    "n_carries", "total_carry_dist",
    "n_shots", "total_xg", "n_dribbles",
    "n_defensive_actions", "n_clearances", "n_interceptions",
    "n_ball_recoveries", "n_blocks",
    "n_miscontrols", "n_dispossessed",
    "n_events_under_pressure", "total_event_count",
]

corr = player_match_df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, ax=ax, annot_kws={"size": 7})
ax.set_title("Feature Correlation Matrix (Per-Player, Per-Match)", fontsize=12)
fig.tight_layout()
plt.show()

## Section 10 — Intensity Profiles by Position

Do different positions show distinct intensity profiles? This helps validate the features are capturing real tactical differences.

In [ ]:
# Map detailed positions to broader roles for cleaner grouping
POSITION_ROLE_MAP = {
    "Goalkeeper": "Goalkeeper",
    "Right Back": "Defender", "Right Wing Back": "Defender",
    "Right Center Back": "Defender", "Center Back": "Defender",
    "Left Center Back": "Defender", "Left Back": "Defender",
    "Left Wing Back": "Defender",
    "Right Defensive Midfield": "Midfielder", "Center Defensive Midfield": "Midfielder",
    "Left Defensive Midfield": "Midfielder",
    "Right Midfield": "Midfielder", "Right Center Midfield": "Midfielder",
    "Center Midfield": "Midfielder", "Left Center Midfield": "Midfielder",
    "Left Midfield": "Midfielder",
    "Right Attacking Midfield": "Midfielder", "Center Attacking Midfield": "Midfielder",
    "Left Attacking Midfield": "Midfielder",
    "Right Wing": "Forward", "Left Wing": "Forward",
    "Right Center Forward": "Forward", "Center Forward": "Forward",
    "Left Center Forward": "Forward", "Striker": "Forward",
    "Secondary Striker": "Forward",
}

player_match_df["position_role"] = player_match_df["primary_position"].map(POSITION_ROLE_MAP)

role_features = ["n_duels", "n_pressures", "n_fouls_won", "n_carries",
                 "n_shots", "n_defensive_actions", "total_carry_dist", "n_dribbles"]

role_means = (
    player_match_df
    .dropna(subset=["position_role"])
    .groupby("position_role")[role_features]
    .mean()
    .round(2)
)

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.flat, role_features):
    data = player_match_df.dropna(subset=["position_role"])
    order = ["Goalkeeper", "Defender", "Midfielder", "Forward"]
    sns.boxplot(data=data, x="position_role", y=col, order=order, ax=ax,
                showfliers=False, palette="Set2")
    ax.set_title(col, fontsize=10)
    ax.set_xlabel("")
fig.suptitle("Intensity Features by Position Role", fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

print("Mean features by position role:")
role_means

## Section 11 — Three-Sixty (360) Coverage Assessment

StatsBomb 360 data provides freeze-frame snapshots of visible player positions around each event. Only a subset of matches have this data — let's quantify the coverage.

In [ ]:
# ── Check 360 coverage ────────────────────────────────────────────────────
threesixty_files = sorted(THREESIXTY_DIR.glob("*.json"))
threesixty_match_ids = {int(f.stem) for f in threesixty_files}

total_matches = player_match_df["match_id"].nunique()
matches_with_360 = len(threesixty_match_ids & set(player_match_df["match_id"].unique()))

print(f"Matches with 360 data: {matches_with_360} / {total_matches} "
      f"({100 * matches_with_360 / total_matches:.1f}%)")
print(f"Total 360 files on disk: {len(threesixty_files)}")

# Which competitions have 360 data?
player_match_df["has_360"] = player_match_df["match_id"].isin(threesixty_match_ids)
coverage_360 = (
    player_match_df.groupby("competition_name")
    .agg(
        n_matches=("match_id", "nunique"),
        n_with_360=("has_360", lambda s: s.groupby(player_match_df.loc[s.index, "match_id"]).first().sum()),
    )
    .assign(pct_360=lambda d: (100 * d["n_with_360"] / d["n_matches"]).round(1))
    .sort_values("n_with_360", ascending=False)
)
coverage_360

## Section 12 — Export Feature Table & Summary

Save the final per-player, per-match feature table and print the full column inventory.

In [ ]:
# ── Column inventory ──────────────────────────────────────────────────────
print("Final feature table columns:\n")
for i, col in enumerate(player_match_df.columns, 1):
    dtype = player_match_df[col].dtype
    non_null = player_match_df[col].notna().mean()
    print(f"  {i:2d}. {col:30s}  {str(dtype):10s}  ({non_null:.1%} non-null)")

print(f"\nShape: {player_match_df.shape[0]:,} rows × {player_match_df.shape[1]} columns")

In [ ]:
# ── Save to CSV ───────────────────────────────────────────────────────────
output_path = Path("../data/statsbomb_player_match_event_features.csv")
player_match_df.to_csv(output_path, index=False)
print(f"Saved: {output_path}  ({player_match_df.shape[0]:,} rows × {player_match_df.shape[1]} cols)")

## Section 13 — Coverage & External Join Memo

### Data Coverage Summary

| Dimension | Value |
|---|---|
| Total event files (matches) | ~3,464 |
| Total player × match rows | (see output above) |
| Unique players | (see output above) |
| Competitions | Multiple (La Liga, Champions League, 1. Bundesliga, World Cup, etc.) |
| Date range | ~2004–2024 (varies by competition) |
| Matches with 360 data | ~326 (sparse, ~9% of total) |

### Feature Coverage Notes

- **Duels, pressures, fouls, carries, shots, defensive actions**: Available for every match (these are core StatsBomb event types). High coverage.  
- **xG (expected goals)**: Only non-null for shots. Sparse per player per match (most players take 0–2 shots).  
- **360 freeze-frame data**: Very sparse (<10% of matches). Not usable as a general feature — only as an optional enrichment for a subset.  
- **Substitution events**: Good coverage. StatsBomb logs every sub with the minute and reason.  
- **Position changes**: Derivable from within-match event positions or lineup `positions` array. Some noise (positions repeat across events even without a real shift).

### External Join Difficulty: StatsBomb → Transfermarkt

**Join key: `player_id`**  
- StatsBomb uses its own internal `player_id` (integer). Transfermarkt also uses its own `player_id`.  
- **There is no shared key.** A fuzzy name-matching pipeline is required.  
- Join candidates: `player_name` (StatsBomb) ↔ `player_name` (Transfermarkt), optionally disambiguated by date of birth, nationality, or team.

**Known challenges:**  
1. **Name variants**: StatsBomb stores full legal names (e.g., "Lionel Andrés Messi Cuccittini"); Transfermarkt stores display names (e.g., "Lionel Messi"). Nicknames, accented characters, and transliterations add noise.  
2. **Team alignment**: StatsBomb team names are not identical to Transfermarkt's (e.g., "Barcelona" vs "FC Barcelona").  
3. **Competition overlap**: StatsBomb open-data covers select competitions/seasons; Transfermarkt covers broadly. The intersection is the usable dataset.  
4. **Temporal alignment**: The injury model needs event features *before* an injury date. This requires knowing which matches a player appeared in, then aggregating features in a lookback window.

**Recommended approach for NB05 (cross-dataset joinability):**  
- Build a player-name crosswalk using fuzzy matching (e.g., `rapidfuzz`) with team name + season as disambiguation.  
- Alternatively, leverage an existing StatsBomb-to-Transfermarkt ID mapping if one can be sourced from community repos.  
- For team names, use a manual mapping table or fuzzy match on competition + team name.

### Feature Usefulness Assessment for Injury Prediction

| Feature group | Expected utility | Rationale |
|---|---|---|
| **Duels (aerial + tackle)** | High | Direct physical contact → trauma / cumulative load |
| **Pressures / counterpresses** | High | High-intensity sprinting / defensive workload |
| **Fouls won (suffered)** | Medium–High | Direct impact events; may precede injury |
| **Carries / carry distance** | Medium | Proxy for running volume with ball |
| **Shots / xG** | Low–Medium | Explosive but infrequent; more match-outcome than load |
| **Defensive actions** | Medium | Clearances, blocks, interceptions = sustained physical effort |
| **Dribbles / dribbled past** | Medium | Change-of-direction load, hamstring stress |
| **Substitution (subbed off)** | Medium | Early sub may indicate in-game issue; late sub = full load |
| **Position diversity** | Low–Medium | Position changes may indicate tactical overload |
| **360 data** | Low (coverage) | Too sparse for general features; possible for case studies |

---

*Next steps → NB05 (cross-dataset joinability): build the player-name crosswalk and attempt a temporal merge with Transfermarkt injury data.*